# Databricks/PySpark for Data Analysts/Engineers — Module 4: Joins in Spark

In Module 3 we joined tables "on the fly", before formally covering `join`. This module
does it properly: join types, semi/anti joins, broadcast joins, and a first look at
reading an execution plan (`explain()`) in the context of joining tables.

## Table of contents
1. [Join types: inner/left/right/outer](#sec1)
2. [Semi and anti joins](#sec2)
3. [Broadcast join and shuffle cost](#sec3)
4. [Reading a join's execution plan](#sec4)
5. [Summary and exercises](#sec5)


We start (or grab the already-running) `SparkSession` -- the entry point to everything in Spark. In Databricks, `spark` is already available automatically in every notebook; the same code also works locally.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("spark_course").getOrCreate()
spark


Let's load our shared dataset -- online store data (`customers`, `products`, `employees`, `orders`, `order_items`), the exact same schema as the SQL course, but this time as Spark DataFrames.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

# --- customers -----------------------------------------------------------
n_customers = 300
cities = ["Warsaw", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segments = ["New", "Standard", "Premium"]
segment_probs = [0.35, 0.45, 0.20]

customers_pd = pd.DataFrame({
    "customer_id": np.arange(1, n_customers + 1),
    "city": rng.choice(cities, size=n_customers, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segments, size=n_customers, p=segment_probs),
    "days_since_signup": rng.integers(1, 900, size=n_customers),
})

# --- products ------------------------------------------------------------
n_products = 40
categories = ["Electronics", "Clothing", "Home", "Sports", "Books"]
avg_prices = {"Electronics": 1200, "Clothing": 150, "Home": 250, "Sports": 300, "Books": 50}

product_categories = rng.choice(categories, size=n_products)
products_pd = pd.DataFrame({
    "product_id": np.arange(1, n_products + 1),
    "category": product_categories,
    "price": [
        round(max(5.0, rng.normal(avg_prices[c], avg_prices[c] * 0.3)), 2)
        for c in product_categories
    ],
})

# --- employees ----------------------------------------------------------
n_employees = 12
regions = ["North", "South", "East", "West"]
employees_pd = pd.DataFrame({
    "employee_id": np.arange(1, n_employees + 1),
    "region": rng.choice(regions, size=n_employees),
    "department": rng.choice(["Sales", "Support"], size=n_employees, p=[0.6, 0.4]),
})

# --- orders ------------------------------------------------------------
n_orders = 3000
order_customer_id = rng.choice(customers_pd["customer_id"], size=n_orders)
order_employee_id = rng.choice(employees_pd["employee_id"], size=n_orders)

segment_map = customers_pd.set_index("customer_id")["segment"]
order_segment = pd.Series(order_customer_id).map(segment_map).values

# cancellation probability depends on customer segment (deliberately built in)
cancel_prob = np.where(
    order_segment == "New", 0.22,
    np.where(order_segment == "Standard", 0.10, 0.04),
)
cancelled = rng.binomial(1, cancel_prob)
order_status = np.where(
    cancelled == 1, "Cancelled",
    rng.choice(["Placed", "Shipped", "Delivered"], size=n_orders, p=[0.2, 0.3, 0.5]),
)

base_date = pd.Timestamp("2024-01-01")
day_offset = rng.integers(0, 730, size=n_orders)

orders_pd = pd.DataFrame({
    "order_id": np.arange(1, n_orders + 1),
    "customer_id": order_customer_id,
    "employee_id": order_employee_id,
    "order_date": [base_date + pd.Timedelta(days=int(d)) for d in day_offset],
    "status": order_status,
})

# --- order_items ------------------------------------------------------
item_rows = []
item_id = 1
product_price = products_pd.set_index("product_id")["price"]

for order_id in orders_pd["order_id"]:
    n_items = rng.integers(1, 5)
    chosen_products = rng.choice(products_pd["product_id"], size=n_items, replace=False)
    for product_id in chosen_products:
        quantity = int(rng.integers(1, 4))
        discount = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        item_rows.append({
            "item_id": item_id,
            "order_id": int(order_id),
            "product_id": int(product_id),
            "quantity": quantity,
            "unit_price": float(product_price[product_id]),
            "discount": float(discount),
        })
        item_id += 1

order_items_pd = pd.DataFrame(item_rows)

# --- convert to Spark DataFrames ---------------------------------------
customers = spark.createDataFrame(customers_pd)
products = spark.createDataFrame(products_pd)
employees = spark.createDataFrame(employees_pd)
orders = spark.createDataFrame(orders_pd)
order_items = spark.createDataFrame(order_items_pd)

customers.show(5)


<a id="sec1"></a>
## 1. Join types: inner/left/right/outer

The syntax is `df1.join(df2, condition, type)` -- exactly the same join types as in SQL
(Module 5 of the SQL course), just now as a DataFrame method. When the joining column has
the same name in both tables, you can pass just that name as a string instead of a full
condition.

In [ ]:
# Default: inner join -- only rows with a match on both sides
orders.join(customers, "customer_id").select(
    "order_id", "customer_id", "city", "segment", "status"
).show(5)


In [ ]:
# left join -- all rows from the left table, even without a match (NULL on the right)
employees.join(orders, "employee_id", "left").select(
    "employee_id", "region", "order_id"
).show(5)


When the joining column names DIFFER between the tables, you have to pass a full
condition as a boolean expression (not a string) -- then both columns stay in the result
(you often need to `.drop()` one of them afterward).

In [ ]:
# Condition as an expression -- useful when column names differ
orders.join(
    customers, orders.customer_id == customers.customer_id, "inner"
).drop(customers.customer_id).select("order_id", "customer_id", "city").show(5)


> ⚠️ **right and full outer -- less common, but good to know**
>
> `"right"` is a left join with the tables swapped (we rarely write it directly -- it's easier to swap the `df1`/`df2` order and use `"left"`). `"outer"` (or `"full"`) returns ALL rows from BOTH sides, with `NULL` wherever there's no match on the other side -- a rare case in practice, but useful for spotting discrepancies between two data sources.

<a id="sec2"></a>
## 2. Semi and anti joins

These are two join types that don't exist under this name in classic SQL (there, you'd
achieve them with `WHERE EXISTS`/`WHERE NOT EXISTS`, as in Module 6 of the SQL course) --
in Spark they're built-in `join()` types. Their key trait: they return columns ONLY from
the left table, even when the condition involves the right one.

In [ ]:
# left_semi -- customers who HAVE at least one order (only customers' columns)
customers.join(orders, "customer_id", "left_semi").show(5)


In [ ]:
# left_anti -- customers who NEVER placed an order (only customers' columns)
customers_without_orders = customers.join(orders, "customer_id", "left_anti")
print(f"Customers without orders: {customers_without_orders.count()}")
customers_without_orders.show(5)


> ⚡ **Semi/anti join instead of join + filter + drop**
>
> Before knowing `left_semi`/`left_anti`, you could get the same result with a regular `join` and filtering rows with (or without) `NULL` on the right side -- but that would require pulling in ALL of the right table's columns, only to discard them right away. Semi/anti join tells Spark upfront that we only care about the left side's columns -- both cleaner code and usually a faster execution plan.

<a id="sec3"></a>
## 3. Broadcast join and shuffle cost

By default, joining two large tables requires a **shuffle** -- reshuffling data across
executors so rows with the same key land on the same machine (**SortMergeJoin**). That's
an expensive network operation. If ONE of the tables is SMALL (fits in a single
executor's memory), Spark can instead send a COPY of it to EVERY executor
(**BroadcastHashJoin**) -- avoiding the shuffle entirely.

In [ ]:
from pyspark.sql import functions as F

# products (40 rows) is a perfect broadcast candidate -- joining with the much
# larger order_items table
joined = order_items.join(F.broadcast(products), "product_id")
joined.select("item_id", "product_id", "category", "price").show(5)


> ⚡ **Spark usually does this automatically**
>
> Spark has a built-in threshold (`spark.sql.autoBroadcastJoinThreshold`, 10 MB by default) -- if one table is smaller, Catalyst will REWRITE the join as a broadcast on its own, with no hint needed from us. `F.broadcast()` is mainly useful when Spark misjudges a table's size (e.g. after a chain of transformations) and wouldn't pick it on its own -- then we hint it explicitly.

> ⚠️ **Broadcasting a large table is a mistake, not an optimization**
>
> Calling `F.broadcast()` on a table that does NOT comfortably fit in a single executor's memory can lead to an out-of-memory error (`OutOfMemoryError`) or a serious slowdown -- this hint only makes sense for genuinely SMALL tables (dictionaries, dimension tables like `products`/`employees`), never for large fact tables like `orders`/`order_items`.

<a id="sec4"></a>
## 4. Reading a join's execution plan

`.explain()` (introduced in Module 1) shows WHICH join type Spark actually chose -- worth
checking whenever a join's performance looks suspicious.

In [ ]:
joined.explain()


In the printed plan, look for `BroadcastHashJoin` (fast, no shuffle) or `SortMergeJoin`
(slower, requires shuffling both sides). We'll come back to reading full execution plans
in detail (physical plan, statistics, stages) in Module 9 (optimization) -- for now it's
enough to recognize these two names.

> 🧱 **The Spark UI in Databricks**
>
> In a real Databricks environment, besides `.explain()` in the notebook, you also have access to the graphical **Spark UI** -- the 'SQL / DataFrame' tab shows the same plans visually, with execution times for every stage. In this course, locally, we rely solely on the text-based `.explain()`.

<a id="sec5"></a>
## 5. Summary and exercises

In this module we covered:
- join types: `inner`, `left`, `right`, `outer` -- the `df1.join(df2, condition, type)` syntax,
- `left_semi` and `left_anti` -- filtering by existence of a match, without pulling in the other side's columns,
- broadcast join (`F.broadcast()`) as a way to avoid an expensive shuffle for small tables,
- reading `.explain()` to tell whether Spark chose `BroadcastHashJoin` or `SortMergeJoin`.

### 📝 Exercise 1: Orders with employee data

Join `orders` with `employees` (inner join on `employee_id`) and display `order_id`, `region`, `department`.

<details>
<summary>Show solution</summary>

```python
orders.join(employees, "employee_id").select(
    "order_id", "region", "department"
).show(5)
```

</details>

### 📝 Exercise 2: Products that were never ordered

Use a `left_anti` join between `products` and `order_items` to find products that never appeared in any order.

<details>
<summary>Show solution</summary>

```python
never_ordered = products.join(order_items, "product_id", "left_anti")
print(f"Products with no orders: {never_ordered.count()}")
never_ordered.show()
```

</details>

### 📝 Exercise 3: Broadcast join with employees

Join `orders` with `F.broadcast(employees)` (employees is a small table) and check in `.explain()` whether Spark used a `BroadcastHashJoin`.

<details>
<summary>Show solution</summary>

```python
from pyspark.sql import functions as F

result = orders.join(F.broadcast(employees), "employee_id")
result.explain()
```

</details>

### 📝 Exercise 4: Premium customers with no Delivered order

Find customers in the `'Premium'` segment who do NOT have any order with status `'Delivered'` (hint: first filter `orders` down to status `'Delivered'`, then do a `left_anti` with `customers`).

<details>
<summary>Show solution</summary>

```python
delivered = orders.filter(orders.status == "Delivered")
premium = customers.filter(customers.segment == "Premium")
premium.join(delivered, "customer_id", "left_anti").show()
```

</details>

> 🔥 **Challenge: a full report with three joins and a broadcast**
>
> Join `orders`, `customers`, and `order_items` (with `F.broadcast(products)` at the end for the category), compute `item_value` (as in Module 2/3), aggregate the sum of value per `(segment, category)`, and print `.explain()` for the whole chain before calling `.show()`.

<details>
<summary>Show solution</summary>

```python
from pyspark.sql import functions as F

joined = (
    orders.join(customers, "customer_id")
    .join(order_items, "order_id")
    .join(F.broadcast(products), "product_id")
    .withColumn("item_value", F.col("quantity") * F.col("unit_price") * (1 - F.col("discount")))
)

result = joined.groupBy("segment", "category").agg(
    F.round(F.sum("item_value"), 2).alias("total_value")
).orderBy(F.col("total_value").desc())

result.explain()
result.show(20)
```

Notice in the plan that only the join with products (broadcast) avoids a shuffle -- the joins with customers and order_items are large tables on both sides, so Spark will still run a SortMergeJoin there.

</details>

## What's next?

In **Module 5** we'll learn Spark SQL -- how to register a DataFrame as a temporary view
(`createOrReplaceTempView`) and write queries exactly like in the SQL course, just now
inside a Spark notebook.